In [1]:
!pip install pyspark datasets huggingface_hub -q

In [2]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')

login(token=hf_token)

In [3]:
!wget -O data.gz "https://go.criteo.net/criteo-research-kaggle-display-advertising-challenge-dataset.tar.gz"

--2026-05-24 06:28:58--  https://go.criteo.net/criteo-research-kaggle-display-advertising-challenge-dataset.tar.gz
Resolving go.criteo.net (go.criteo.net)... 74.119.117.38, 2620:100:a00b::27
Connecting to go.criteo.net (go.criteo.net)|74.119.117.38|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://criteostorage.blob.core.windows.net/criteo-research-datasets/kaggle-display-advertising-challenge-dataset.tar.gz [following]
--2026-05-24 06:28:59--  https://criteostorage.blob.core.windows.net/criteo-research-datasets/kaggle-display-advertising-challenge-dataset.tar.gz
Resolving criteostorage.blob.core.windows.net (criteostorage.blob.core.windows.net)... 20.209.1.1
Connecting to criteostorage.blob.core.windows.net (criteostorage.blob.core.windows.net)|20.209.1.1|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4576820670 (4.3G) [application/x-gzip]
Saving to: ‘data.gz’

data.gz             100%[===================>]   4.26G  32.8M

In [4]:
!tar -zxvf /content/data.gz

tar: Ignoring unknown extended header keyword 'SCHILY.dev'
tar: Ignoring unknown extended header keyword 'SCHILY.ino'
tar: Ignoring unknown extended header keyword 'SCHILY.nlink'
readme.txt
tar: Ignoring unknown extended header keyword 'LIBARCHIVE.creationtime'
tar: Ignoring unknown extended header keyword 'SCHILY.dev'
tar: Ignoring unknown extended header keyword 'SCHILY.ino'
tar: Ignoring unknown extended header keyword 'SCHILY.nlink'
test.txt
tar: Ignoring unknown extended header keyword 'SCHILY.dev'
tar: Ignoring unknown extended header keyword 'SCHILY.ino'
tar: Ignoring unknown extended header keyword 'SCHILY.nlink'
train.txt


In [5]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Criteo_MMDS_Benchmarking") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "50") \
    .config("spark.driver.maxResultSize", "2g") \
    .getOrCreate()

print("Spark Version:", spark.version)

Spark Version: 4.0.2


In [6]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

fields = [StructField("label", IntegerType(), True)]

for i in range(1, 14):
    fields.append(StructField(f"I{i}", IntegerType(), True))

for i in range(1, 27):
    fields.append(StructField(f"C{i}", StringType(), True))

criteo_schema = StructType(fields)

print(len(criteo_schema.fields))

40


In [7]:
# df_raw = spark.read.csv(f"{raw_data_dir}/*.gz", sep="\t", schema=criteo_schema)
df_raw = spark.read.csv("train.txt", sep="\t", schema=criteo_schema)
total_count = df_raw.count()
print(f"Num records: {total_count}")

Num records: 45840617


In [ ]:
from pyspark.ml.feature import Imputer
from pyspark.sql.functions import col, log1p, when, monotonically_increasing_id
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# Gán index theo thứ tự gốc
df_raw = df_raw.withColumn("row_id", monotonically_increasing_id())

window_spec = Window.orderBy("row_id")

df_indexed = df_raw.withColumn("row_num", row_number().over(window_spec))

df_indexed.cache()

train_end = int(total_count * 0.8)
val_end   = int(total_count * 0.9)

# Chia tập — giữ nguyên thứ tự gốc
train_df = df_indexed.filter(col("row_num") <= train_end).drop("row_num", "row_id")
val_df   = df_indexed.filter((col("row_num") > train_end) & (col("row_num") <= val_end)).drop("row_num", "row_id")
test_df  = df_indexed.filter(col("row_num") > val_end).drop("row_num", "row_id")

num_cols = [f"I{i}" for i in range(1, 14)]
cat_cols = [f"C{i}" for i in range(1, 27)]

# Fit imputer chỉ trên train
imputer = Imputer(inputCols=num_cols, outputCols=num_cols, strategy="median")
imputer_model = imputer.fit(train_df)

def preprocess(df, fitted_imputer_model):
    # Fill categorical null
    fill_dict = {c: f"unknown_{c.lower()}" for c in cat_cols}
    df = df.fillna(fill_dict)

    # Dùng model đã fit từ train
    df = fitted_imputer_model.transform(df)

    # Clamp âm + log1p
    for c in num_cols:
        df = df.withColumn(c, when(col(c) < 0, 0.0).otherwise(col(c)))
        df = df.withColumn(c, log1p(col(c)))

    return df

train_clean = preprocess(train_df, imputer_model)
val_clean   = preprocess(val_df,   imputer_model)
test_clean  = preprocess(test_df,  imputer_model)

print("Train:", train_clean.count())
print("Val:",   val_clean.count())
print("Test:",  test_clean.count())

train_clean.show(10)

Train: 3667653
Val: 4586790
Test: 4590134

+-----+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+--------+--------+--------+--------+--------+---------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+--------+----------+----------+--------+----------+--------+--------+----------+----------+
|label|                I1|                I2|                I3|                I4|                I5|                I6|                I7|                I8|                I9|               I10|               I11|               I12|               I13|      C1|      C2|      C3|      C4|      C5|       C6|      C7|      C8|      C9|     C10|     C11|     C12|     C13|     C14|     C15|     C16|     C17|     C18|       C19|       C20|     C21|       C2

In [9]:
output_dir = "/content/criteo_splits"

train_path = f"{output_dir}/train"
val_path = f"{output_dir}/validation"
test_path = f"{output_dir}/test"

train_clean.coalesce(50).write.mode("overwrite").parquet(train_path)
val_clean.coalesce(4).write.mode("overwrite").parquet(val_path)
test_clean.coalesce(4).write.mode("overwrite").parquet(test_path)

print("Đã hoàn tất chuyển đổi và lưu file Parquet!")

Đã hoàn tất chuyển đổi và lưu file Parquet!


In [ ]:
from datasets import load_dataset, DatasetDict
print("Đang chuyển đổi dữ liệu sang định dạng Hugging Face...")
dataset_dict = DatasetDict({
    "train": load_dataset(
        "parquet",
        data_files=f"{train_path}/part-*.parquet",
        split="train"
    ),

    "validation": load_dataset(
        "parquet",
        data_files=f"{val_path}/part-*.parquet",
        split="train"
    ),

    "test": load_dataset(
        "parquet",
        data_files=f"{test_path}/part-*.parquet",
        split="train"
    )
})

repo_id = "nmpogg/criteo-cleaned-v2"

dataset_dict.push_to_hub(repo_id)

print(f"Đã đẩy thành công cả 3 tập (train, val, test) lên Hugging Face: {repo_id}")

Đang chuyển đổi dữ liệu sang định dạng Hugging Face...


Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

Uploading the dataset shards:   0%|          | 0/40 [00:00<?, ? shards/s]

Creating parquet from Arrow format:   0%|          | 0/1147 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

...

Creating parquet from Arrow format:   0%|          | 0/1148 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

                              :   2%|2         | 3.76MB /  153MB            

Đã đẩy thành công cả 3 tập (train, val, test) lên Hugging Face: nmpogg/criteo-cleand-v2
